# CSE-4632 Digital Signal Processing Lab — Lab 5

**Task 1:** Read the tuning-fork recording, plot it, and find its ringing frequency using `numpy.fft.rfft`.

**Task 2:** Implement the forward DFT and inverse DFT from scratch (Tables 8-2 and 8-1, Smith DSP Guide), and verify against the Figure 8-6 example.


## Task 1: Spectral Analysis of a Tuning Fork Recording

Place `Lab 4.wav` in the same directory as this notebook before running this section.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

# --- Read the audio file ---
fs, data = wavfile.read('Lab 4.wav')

# If the file is stereo, keep only one channel
if data.ndim > 1:
    data = data[:, 0]

data = data.astype(np.float64)
N = len(data)
t = np.arange(N) / fs

print(f"Sampling rate     : {fs} Hz")
print(f"Number of samples : {N}")
print(f"Duration          : {N/fs:.3f} s")

### Plot the time-domain signal

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, data)
plt.title('Tuning Fork Recording - Time Domain')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

### Spectral analysis using `numpy.fft.rfft`

In [ ]:
X = np.fft.rfft(data)
freqs = np.fft.rfftfreq(N, d=1.0 / fs)
magnitude = np.abs(X)

plt.figure(figsize=(10, 4))
plt.plot(freqs, magnitude)
plt.title('Magnitude Spectrum (rfft)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.xlim(0, 5000)  # tuning forks usually ring below a few kHz
plt.tight_layout()
plt.show()

In [ ]:
# --- Find the dominant frequency (ignore DC bin 0) ---
peak_index = np.argmax(magnitude[1:]) + 1
peak_freq = freqs[peak_index]

print(f"Dominant frequency : {peak_freq:.2f} Hz")

## Task 2: Forward DFT and Inverse DFT From Scratch

Implemented using only `math`/`numpy` basics (no high-level FFT calls), following:
- **Table 8-2, p.160** (Analysis / Forward DFT, Eq. 8-4)
- **Table 8-1, p.153** (Synthesis / Inverse DFT, Eq. 8-2 & 8-3)

Verified against the worked example in **Figure 8-6, p.155** of *The Scientist and Engineer's Guide to DSP* (Smith): an impulse `x[0] = 32` (N = 32) should give `ReX[k] = 32` for all k and `ImX[k] = 0`.


In [ ]:
import math

def myDFT(x):
    """
    Forward DFT (real DFT), following Table 8-2 / Eq. 8-4.
    x   : time-domain signal, length N
    returns (ReX, ImX), each of length N/2 + 1
    """
    N = len(x)
    half = N // 2
    ReX = [0.0] * (half + 1)
    ImX = [0.0] * (half + 1)

    for k in range(half + 1):
        for i in range(N):
            angle = 2 * math.pi * k * i / N
            ReX[k] += x[i] * math.cos(angle)
            ImX[k] -= x[i] * math.sin(angle)

    return ReX, ImX


def myIDFT(ReX, ImX):
    """
    Inverse DFT (real DFT), following Table 8-1 / Eq. 8-2, 8-3.
    ReX, ImX : frequency-domain arrays, each length N/2 + 1
    returns  : reconstructed time-domain signal x, length N
    """
    half = len(ReX) - 1
    N = 2 * half

    # Eq. 8-3: convert frequency domain values into sinusoid amplitudes
    ReXbar = [ReX[k] / (N / 2) for k in range(half + 1)]
    ImXbar = [-ImX[k] / (N / 2) for k in range(half + 1)]
    ReXbar[0] = ReX[0] / N
    ReXbar[half] = ReX[half] / N

    # Eq. 8-2: synthesis
    x = [0.0] * N
    for i in range(N):
        total = 0.0
        for k in range(half + 1):
            angle = 2 * math.pi * k * i / N
            total += ReXbar[k] * math.cos(angle)
            total += ImXbar[k] * math.sin(angle)
        x[i] = total

    return x

### Verification using the Figure 8-6 example

In [ ]:
# Time-domain signal: an impulse at sample 0 with amplitude 32,
# N = 32 points, all other samples zero.
N = 32
x = [0.0] * N
x[0] = 32.0

ReX, ImX = myDFT(x)

print("Forward DFT result (should match Fig. 8-6b: ReX = 32 for all k, ImX = 0):")
print("ReX:", [round(v, 4) for v in ReX])
print("ImX:", [round(v, 4) for v in ImX])

In [ ]:
x_reconstructed = myIDFT(ReX, ImX)
print("Inverse DFT result (should match Fig. 8-6a: impulse of 32 at i=0):")
print("x  :", [round(v, 4) for v in x_reconstructed])

max_error = max(abs(a - b) for a, b in zip(x, x_reconstructed))
print(f"\nMax reconstruction error: {max_error:.2e}")

In [ ]:
# Sanity check against numpy's FFT-based rfft
# (not used inside myDFT/myIDFT themselves — numpy doesn't negate the imaginary part,
# so we flip the sign to match the book's convention)
import numpy as np

X_np = np.fft.rfft(x)
print("numpy rfft ReX (sanity check):", np.round(X_np.real, 4))
print("numpy rfft ImX (sanity check):", np.round(-X_np.imag, 4))